In [ ]:

%load_ext autoreload
%autoreload 2

import ezy_seq as ezy

import scanpy as sc
import pandas as pd
import os
from pathlib import Path
import matplotlib.pyplot as plt


import numpy as np

import squidpy as sq
import anndata as ad

In [ ]:
import pandas as pd
import numpy as np
import napari
import os
from matplotlib import Path
def load_polys(fp):
    if not os.path.exists(fp): return []
    df = pd.read_csv(fp)
    # pick axis columns (accept 'axis-0'/'axis-1' or first two 'axis*' matches)
    axes = ['axis-0','axis-1']
    xcol,ycol = axes[1], axes[0]
    df = df.dropna(subset=[xcol,ycol])
    return [g[[ycol,xcol]].to_numpy() for _,g in df.groupby('index')]


def load_all_polygons(project_folder):

    project_folder_ = Path(project_folder)
    polygons = {}
    print('here')
    for fp in project_folder_.rglob("*-polygons.csv.gz"):
        slide_name_ = fp.stem.replace("-polygons", "")
        slide_name = slide_name_.replace(".csv", "")
        polygons[slide_name] = pd.read_csv(fp)
    return polygons


# Example usage:
project_folder = r"E:\Carter-Woods\CosMx export\10_31_export"
polygons_dict = load_all_polygons(project_folder)


sample_list=[['120L','323DY'],['368R'],['1182L','118B'],['217R','144_2L'],['367R','215L']]
layer_folders=['120Land323DY','368R','118Band1182L','217Rand144_2L','367Rand215L']
polygons=[]
for folder in layer_folders:
    polygons.append(polygons_dict[folder])


In [2]:
adata_full=sc.read_h5ad(r"/path/to/adata.h5ad")
#quint_adata=sc.read_h5ad(r"/path/to/quint_updated.h5ad")
#adata_old=sc.read_h5ad(r"/path/to/quint_updated.h5ad")


In [ ]:
adata_full.obs['quint_region'].value_counts()

In [ ]:
import pandas as pd
import numpy as np
import napari
import os
from pathlib import Path  # FIXED: Use pathlib for file paths
import imageio.v2 as imageio
import matplotlib.pyplot as plt

# --- Helper Functions ---

def load_polys(fp):
    """
    Loads polygons from a CSV file formatted for Napari (index, axis-0, axis-1).
    Returns a list of (N, 2) numpy arrays.
    """
    if not os.path.exists(fp): 
        print(f"File not found: {fp}")
        return []
    
    try:
        df = pd.read_csv(fp)
        # Check if standard Napari export columns exist
        if 'axis-0' in df.columns and 'axis-1' in df.columns:
            # Napari uses (Y, X) coordinate system. 
            # axis-0 is usually Y, axis-1 is X.
            # We return (Y, X) for Napari.
            ycol, xcol = 'axis-0', 'axis-1'
        else:
            print(f"Columns 'axis-0'/'axis-1' not found in {fp}. Checking for alternatives...")
            return []

        df = df.dropna(subset=[xcol, ycol])
        
        # Group by 'index' to separate distinct polygons (shapes)
        return [g[[ycol, xcol]].to_numpy() for _, g in df.groupby('index')]
        
    except Exception as e:
        print(f"Error loading {fp}: {e}")
        return []

def load_all_polygons(project_folder):
    """Loads CosMx cell segmentation polygons."""
    project_folder_ = Path(project_folder)
    polygons = {}
    for fp in project_folder_.rglob("*-polygons.csv.gz"):
        slide_name_ = fp.stem.replace("-polygons", "")
        slide_name = slide_name_.replace(".csv", "")
        polygons[slide_name] = pd.read_csv(fp)
    return polygons

# --- Setup Paths & Data ---

# Example usage:
project_folder = r"E:\Carter-Woods\CosMx export\10_31_export"
# polygons_dict = load_all_polygons(project_folder) # Uncomment if you need the cell polygons

base = r"/path/to/project"

sample_list = [['120L','323DY'],['368R'],['1182L','118B'],['217R','144_2L'],['367R','215L']]
layer_folders = ['120Land323DY','368R','118Band1182L','217Rand144_2L','367Rand215L']

# Define region mapping: Name -> File Name (so you can easily add more later)
regions_to_load = {
    'Hippocampus': 'Hippocampus.csv',
    'Cortex': 'Cortex.csv',
    'Cerebellum': 'Cerebellum.csv',
    'Olfactory': 'Olfactory.csv',
    'Striatum': 'Striatum.csv'
}


# Define distinct colors for regions
region_colors = {
    'Hippocampus': 'magenta',
    'Cortex': 'cyan',
    'Cerebellum': 'yellow',
    'Olfactory': 'green',
    'Striatum': 'orange'
}

# --- Main Visualization Loop ---

# Assuming 'adata_full' is already loaded in your environment
# for samples, polygon_df, slide_folder in zip(sample_list, polygons, layer_folders): # Original
for samples, slide_folder in zip(sample_list, layer_folders):
    
    # Filter Anndata for current sample
    s_df = adata_full[adata_full.obs['sample_ID'].isin(samples)].copy()
    
    # --- 1. Setup Colors for Cells ---
    unique_clusters = s_df.obs['Rerun_w_CosMx_Profile_Neighbor.network.expression.space.1_1_cluster_Rerun_w_CosMx_Profile_Leiden.Clustering.1_1'].unique()
    cmap = plt.get_cmap('tab20') 
    colors_list = [cmap(i) for i in np.linspace(0, 1, len(unique_clusters))]
    cluster_color_dict = dict(zip(unique_clusters, colors_list))
    assigned_colors = s_df.obs['Rerun_w_CosMx_Profile_Neighbor.network.expression.space.1_1_cluster_Rerun_w_CosMx_Profile_Leiden.Clustering.1_1'].map(cluster_color_dict).tolist()

    # --- 2. Initialize Viewer ---
    viewer = napari.Viewer(title=f"Slide: {slide_folder}")

    # Add All CosMx Cells
    viewer.add_points(
        s_df.obsm['spatial_fov'], 
        size=100,
        border_width=0,
        face_color=assigned_colors,
        name="CosMx cells"
    )

    # --- 4. Load & Add Brain Region Polygons ---
    print(f"Loading regions for {slide_folder}...")
    
    for region_name, file_name in regions_to_load.items():
        fp = os.path.join(base, slide_folder, file_name)
        
        polys = load_polys(fp)
        
        if polys:
            print(f"  -> Adding {region_name} ({len(polys)} shapes)")
            viewer.add_shapes(
                polys, 
                shape_type='polygon', 
                name=region_name, 
                edge_width=10,        # Make edges visible
                edge_color=region_colors.get(region_name, 'white'),
                face_color=[1,1,1,1], # Transparent face to see cells underneath
                opacity=0.8
            )
        else:
            pass # Use pass to keep console clean, or print if debugging

    napari.run()

In [ ]:
list(adata_full.obs.columns)

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import scanpy as sc
from matplotlib.path import Path

# adata_full=sc.read_h5ad(r"PATH_TO_YOUR_FILE")
base=r"/path/to/project"

dfs = []
for samples, polygon_df, slide_folder in zip(sample_list, polygons, layer_folders):
    # Create the subset for the current slide/batch
    s_df = adata_full[adata_full.obs['sample_ID'].isin(samples)].copy()
    
    # Set up coordinates (Y, X)
    cell_xy = np.column_stack((s_df.obsm['spatial_fov'][:, 1], s_df.obsm['spatial_fov'][:, 0]))
    
    # Initialize the new sample_ID column
    # We start with 'Unassigned' so we can see which cells didn't fall into a polygon
    new_sample_ids = pd.Series('Unassigned', index=s_df.obs.index, dtype=str)

    # 1. Find all CSV files in the current slide folder
    search_path = os.path.join(base, slide_folder, "*.csv")
    found_csvs = glob.glob(search_path)
    
    if not found_csvs:
        print(f"No CSVs found in {slide_folder}")

    for fp in found_csvs:
        # 2. Extract the 'sample code' from the filename (e.g. 'Sample_A.csv' -> 'Sample_A')
        file_name = os.path.basename(fp)
        sample_code = os.path.splitext(file_name)[0]
        if sample_code!='Striatum':
            continue

        print(sample_code)

        df = pd.read_csv(fp)
        
        # detect axis columns (prefer 'axis-1' (x) and 'axis-0' (y))
        xcol = 'axis-1' if 'axis-1' in df.columns else None
        ycol = 'axis-0' if 'axis-0' in df.columns else None
        
        if xcol is None or ycol is None:
            print(f"bad cols in {fp}; need axis-0/axis-1")
            continue

        # group vertices by polygon index and test points
        hit_mask = np.zeros(len(cell_xy), dtype=bool)
        
        for _, g in df.groupby('index'):
            verts = np.column_stack((g[xcol].to_numpy(dtype=float), g[ycol].to_numpy(dtype=float)))
            if len(verts) < 3:  # skip degenerate
                continue
            p = Path(verts)
            # update mask: true if point is in ANY of the polygons for this sample code
            hit_mask |= p.contains_points(cell_xy)

        # 3. Assign the filename (sample_code) to the cells inside the polygon
        if hit_mask.any():
            new_sample_ids.loc[s_df.obs.index[hit_mask]] = sample_code
        
        print(f"Assigned '{sample_code}': {hit_mask.sum()} cells")

    # Overwrite/Assign the sample_ID column
    s_df.obs['napari_region'] = new_sample_ids
    dfs.append(s_df)
    
    print(f"finished {slide_folder} — sample_ID distribution:\n{s_df.obs['sample_ID'].value_counts()}\n")

adata_full = sc.concat(dfs, join='outer', axis=0, merge="first", index_unique='-')

#adata_full.write_h5ad(r"/path/to/adata.h5ad")

In [35]:
adata_full.write_h5ad(r"/path/to/adata.h5ad")


In [ ]:
base_path=r"/path/to/project"
import matplotlib.path as mplPath
from pathlib import Path
for sample_id, slide_folder in zip(sample_list, layer_folders):
    
    # 1. Look specifically for the Striatum file
    striatum_path = os.path.join(base_path, slide_folder, "Striatum.csv")
    
    if not os.path.exists(striatum_path):
        print(f"No Striatum.csv found for {sample_id}")
        continue

    # 2. Get coordinates ONLY for the current sample
    # Create a mask for the current sample's cells
    # GOOD (Checks if the cell's ID is in the list of IDs)
    sample_mask = adata_full.obs['sample_ID'].isin(sample_id)
    
    # Extract Y, X coordinates (keeping your original Y=1, X=0 logic)
    subset_coords = adata_full[sample_mask].obsm['spatial_fov']
    cell_xy = np.column_stack((subset_coords[:, 1], subset_coords[:, 0]))

    # 3. Process the Polygon CSV
    df = pd.read_csv(striatum_path)
    
    xcol = 'axis-1' if 'axis-1' in df.columns else None
    ycol = 'axis-0' if 'axis-0' in df.columns else None
    
    if not xcol or not ycol:
        print(f"Skipping {sample_id}: CSV missing axis columns.")
        continue

    # Create a boolean mask for cells inside the polygon(s)
    is_in_striatum = np.zeros(len(cell_xy), dtype=bool)

    for _, g in df.groupby('index'):
        verts = np.column_stack((g[xcol], g[ycol]))
        if len(verts) >= 3:
            path = mplPath.Path(verts)
            is_in_striatum |= path.contains_points(cell_xy)

    # 4. TARGETED UPDATE
    # Only update if we actually found cells, and only update THOSE specific cells.
    if is_in_striatum.any():
        # Map the boolean mask back to the global AnnData indices
        hit_indices = adata_full.obs.index[sample_mask][is_in_striatum]
        
        # Overwrite only these cells with 'Striatum'
        # Existing labels for other cells remain untouched
        # Unlock the column by converting it to string
    if "Striatum" not in adata_full.obs['napari_region'].cat.categories:
        adata_full.obs['napari_region'] = adata_full.obs['napari_region'].cat.add_categories("Striatum")
        
    print(f"Updated {len(hit_indices)} cells to 'Striatum' in {sample_id}")

print("\nUpdate complete.")

In [3]:
adata_full=sc.read_h5ad(r"/path/to/napari_updated.h5ad")
#adata_full=adata_full1[adata_full1.obs['napari_region']!='Brain_Stem']
#adata_full.obs['napari_region'].unique()

In [ ]:
import matplotlib.pyplot as plt
cell_type_of_interest = "Sncg"   # <-- change this
group_col = "FMT"                         # e.g. "FMT", "sample_ID", "region"
celltype_col = "cell_type"

orientation = "vertical"                    # "vertical" or "horizontal"
colors = (
    (85/255, 160/255, 251/255),             # neg_clr (unused, but kept for symmetry)
    (249/255, 100/255, 149/255)             # pos_clr
)
variable = "percent"
# ===================================================
df = adata_full[(adata_full.obs['FMT']!='Cntrl') & (adata_full.obs['napari_region']!='Brain_Stem')].obs.copy()

#df = adata_full[(adata_full.obs['FMT']!='Cntrl')].obs.copy()


# ---- compute % abundance ----
total_counts = (
    df.groupby(group_col)[celltype_col]
      .count()
      .rename("total_cells")
)
target_counts = (
    df[df[celltype_col] == cell_type_of_interest]
      .groupby(group_col)[celltype_col]
      .count()
      .rename("target_cells")
)
abundance = pd.concat([total_counts, target_counts], axis=1).fillna(0)
abundance[variable] = (abundance["target_cells"] / abundance["total_cells"]) * 100
abundance = abundance.reindex(abundance[variable].sort_values(ascending=False).index)

# ---- assign colors by FMT group ----
unique_groups = abundance.index
palette = plt.cm.tab10.colors  # use 'Set2', 'Paired', etc. for other looks
color_map = {grp: palette[i % len(palette)] for i, grp in enumerate(unique_groups)}
bar_colors = [color_map[g] for g in abundance.index]

# ---- PLOT (vertical) ----
plt.figure(figsize=(2,6))
plt.bar(abundance.index, abundance[variable],
        color=bar_colors, edgecolor='black')

plt.axhline(0, color="black", linewidth=1)
#plt.ylabel("% of cells")
plt.xlabel(group_col)
#plt.title(f"{cell_type_of_interest} abundance by {group_col}")
plt.xticks(rotation=45, ha="right")
plt.tick_params(size=14)
plt.tight_layout()
plt.ylim(0,5)
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# --- user settings ---
cell_type_of_interest = "Sncg"   # change this
percent_is_fraction = False                 # True if 0–1, False if already 0–100
# ----------------------

# Filter to just the cell type you care about
sub = df[df["cell_type"] == cell_type_of_interest].copy()

# If percent is stored as a fraction 0–1, convert to %
if percent_is_fraction:
    sub["percent"] = sub["percent"] * 100

# Sort bars (optional, looks nicer)
sub = sub.sort_values("percent", ascending=False)

plt.figure(figsize=(6, 4))
plt.bar(sub["group"], sub["percent"])

plt.ylabel("% of cells")
plt.xlabel("Group")
plt.title(f"{cell_type_of_interest} abundance by group")

plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
sc.pl.umap(adata_full, color="cell_type")

In [ ]:
sc.pl.umap(adata_full[(adata_full.obs['napari_region']=='Hippocampus') | (adata_full.obs['tissue']=='Hippocampus')], color="cell_type",palette=plt.cm.tab10.colors)
sc.pl.umap(adata_full[(adata_full.obs['napari_region']=='Hippocampus') ], color="cell_type",palette=plt.cm.tab10.colors)
sc.pl.umap(adata_full[ adata_full.obs['tissue']=='Hippocampus'], color="cell_type",palette=plt.cm.tab10.colors)

In [ ]:
from typing import Dict, Union
import numpy as np
import pandas as pd
from anndata import AnnData
import re

# ---------------------------------------------------------
# 1. DATA CLEANING & BROAD REGION DEFINITION
# ---------------------------------------------------------

# A. Remove cells where 'quint_region' is a coordinate string (e.g., "0,0,0")
#    We assume "coordinate strings" are composed of digits and commas.
#    This regex looks for: Digit(s) + Comma + Digit(s) + Comma + Digit(s)
coordinate_pattern = r'^\d+,\d+,\d+$'
mask_coords = adata_full.obs['quint_region'].astype(str).str.match(coordinate_pattern)
adata_full = adata_full[~mask_coords].copy()

# B. Remove Olfactory (as requested previously)
adata_full = adata_full[adata_full.obs['napari_region'] != 'Olfactory'].copy()

# C. Create 'broad_quint_region' column
#    Logic: If quint_region starts with "Cortex", label as "Cortex".
#           Otherwise, keep the original quint_region label.
def simplify_region(val):
    val_str = str(val)
    if val_str.startswith("Cortex"):
        return "Cortex"
    return val_str

adata_full.obs['broad_quint_region'] = adata_full.obs['quint_region'].apply(simplify_region)

# ---------------------------------------------------------
# 2. SAMPLING FUNCTIONS (Unchanged logic, applies to new col)
# ---------------------------------------------------------

random_state = 0

def alloc_by_baseline_with_caps(baseline_counts: pd.Series, caps: pd.Series, total: int) -> pd.Series:
    """
    Allocate 'total' items across indices in proportion to 'baseline_counts',
    without exceeding 'caps' (remaining available per region).
    """
    out = pd.Series(0, index=baseline_counts.index, dtype=int)
    remain = int(total)
    w = baseline_counts.clip(lower=0).astype(float)
    if w.sum() == 0 or remain <= 0:
        return out

    # 1st pass: proportional
    raw = w / w.sum() * remain
    base = np.floor(raw).astype(int)

    # respect caps
    take = base.clip(upper=caps.astype(int))
    out += take
    remain -= int(take.sum())

    # 2nd pass: give leftover to largest fractional remainders
    if remain > 0:
        frac = (raw - base).sort_values(ascending=False)
        for idx in frac.index:
            if remain == 0:
                break
            if out[idx] < int(caps.get(idx, 0)):
                out[idx] += 1
                remain -= 1
    return out

def set_region_abundance_by_FMT(
    adata: AnnData,
    target_by_fmt: Dict[str, Dict[str, Union[int, float]]],
    total_per_fmt: Union[int, Dict[str, int], None] = None,
    region_col: str = "broad_quint_region",  # DEFAULT CHANGED TO NEW COL
    fmt_col: str = "FMT",
    random_state: int | None = 0,
) -> AnnData:
    """
    Select cells per FMT to match targets for specified regions.
    Uses 'broad_quint_region' to handle aggregated Cortex targets.
    """
    rng = np.random.default_rng(random_state)

    obs_rf = adata.obs[[region_col, fmt_col]].dropna().copy()
    avail = obs_rf.groupby([fmt_col, region_col]).size().unstack(fill_value=0)

    # Internal helper reused
    def _alloc_by_baseline_with_caps(baseline, caps, total):
        return alloc_by_baseline_with_caps(baseline, caps, total)

    selected = []

    for fmt in avail.index:
        row = avail.loc[fmt]
        total_avail = int(row.sum())

        if isinstance(total_per_fmt, dict):
            t = int(min(total_per_fmt.get(fmt, total_avail), total_avail))
        elif isinstance(total_per_fmt, int):
            t = int(min(total_per_fmt, total_avail))
        else:
            t = total_avail

        baseline = (row / row.sum()).fillna(0)
        desired = pd.Series(0, index=row.index, dtype=int)

        if fmt in target_by_fmt:
            tgt = target_by_fmt[fmt]
            is_count = all(isinstance(v, (int, np.integer)) for v in tgt.values())

            if is_count:
                for r, v in tgt.items():
                    if r in desired.index:
                        desired[r] = int(v)
            else:
                frac_map = {r: float(v) for r, v in tgt.items() if r in desired.index}
                sfrac = sum(frac_map.values())
                if sfrac > 1.0 and sfrac > 0:
                    frac_map = {r: v / sfrac for r, v in frac_map.items()}
                for r, fr in frac_map.items():
                    desired[r] = int(round(fr * t))

            desired = desired.clip(upper=row)
            unspecified = [r for r in desired.index if r not in tgt]
            remain = max(0, t - int(desired.sum()))
            
            if unspecified and remain > 0:
                caps_unspec = (row - desired).loc[unspecified].clip(lower=0)
                alloc = _alloc_by_baseline_with_caps(baseline.loc[unspecified], caps_unspec, remain)
                desired.loc[unspecified] += alloc

        else:
            caps_all = row.clip(lower=0)
            desired = _alloc_by_baseline_with_caps(row, caps_all, t)

        while int(desired.sum()) > t:
            biggest = desired.idxmax()
            if desired[biggest] == 0: break
            desired[biggest] -= 1

        desired = desired.clip(upper=row)
        sub = obs_rf[obs_rf[fmt_col] == fmt]
        
        for region, need in desired.items():
            n = int(need)
            if n <= 0: continue
            pool = sub.index[sub[region_col] == region].to_numpy()
            if n >= len(pool):
                selected.extend(pool.tolist())
            else:
                selected.extend(rng.choice(pool, size=n, replace=False).tolist())

    sel_set = set(selected)
    sel_index = [idx for idx in obs_rf.index if idx in sel_set]
    if not sel_index:
        raise ValueError("No cells selected.")

    return adata[sel_index].copy()

# ---------------------------------------------------------
# 3. STATS CALCULATION (Using 'broad_quint_region')
# ---------------------------------------------------------

sample_col = "sample_ID"
region_col = "broad_quint_region"  # <--- CHANGED: Now using the broad grouping

# Counts per sample per region
counts_per_sample = adata_full.obs.groupby([sample_col, region_col]).size().unstack(fill_value=0)
# Convert to fractions
fractions_per_sample = counts_per_sample.div(counts_per_sample.sum(axis=1), axis=0)

# Calculate Mean and STD
region_means = fractions_per_sample.mean()
region_stds = fractions_per_sample.std()

# Define Cortex Stats (Consolidated)
cortex_n = region_means.get("Cortex", 0.0)
cortex_h = cortex_n + region_stds.get("Cortex", 0.0)
cortex_l = max(0.0, cortex_n - region_stds.get("Cortex", 0.0))

print(cortex_h)
print(cortex_l)

mean_cort = region_stds.get("Cortex", 0.0)
print("--- Calculated Stats (Fractions) ---")
print(f"Cortex:      Mean={cortex_n:.4f}, STD={mean_cort:.4f}, High={cortex_h:.4f}, Low={cortex_l:.4f}")

# ---------------------------------------------------------
# 4. TARGET DEFINITIONS & EXECUTION
# ---------------------------------------------------------

# Note: We now target "Cortex" (the broad category)
target_by_Cortex_up = {
    "Stroke_FMT": {"Cortex": cortex_h},  
    "Healthy_FMT": {"Cortex": cortex_l},
    "Cntrl": {"Cortex": cortex_n},   
}
target_by_Cortex_down = {
    "Stroke_FMT": {"Cortex": cortex_l},   
    "Healthy_FMT": {"Cortex": cortex_h},
    "Cntrl": {"Cortex": cortex_n}, 
}

print(f"Target High: {cortex_h}")
print(f"Target Norm: {cortex_n}")
print(f"Target Low:  {cortex_l}")

# Run Sampling using the new 'broad_quint_region' column explicitly
cortex_up_q = set_region_abundance_by_FMT(
    adata_full, 
    target_by_Cortex_up, 
    total_per_fmt=125000, 
    region_col="broad_quint_region", # Ensure we use the broad col
    random_state=0
)

cortex_down_q = set_region_abundance_by_FMT(
    adata_full, 
    target_by_Cortex_down, 
    total_per_fmt=125000, 
    region_col="broad_quint_region", # Ensure we use the broad col
    random_state=0
)

In [ ]:
#adata_full.obs['quint_region'].value_counts()
#adata_full.obs['broad_quint_region'].value_counts()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import pandas as pd
import numpy as np

# 1. CONFIGURATION
fmt_col = "FMT" 
region_col = "quint_region"
napari_col = "napari_region" # New reference for the grouping logic

# FIX: Access .obs for filtering
# Remove "coordinate-style" labels (e.g., "0,0,0")
coordinate_pattern = r'^\d+,\d+,\d+$'
mask_coords = adata_full.obs[region_col].astype(str).str.match(coordinate_pattern)
adata_full_clean = adata_full[~mask_coords].copy()

# 2. DEFINE GROUPING LOGIC
# Requirement: If napari_region is 'Cortex', use 'Cortex'. 
#              Otherwise, use the name from quint_region.
adata_full_clean.obs['grouped_region'] = np.where(
    adata_full_clean.obs[napari_col] == 'Cortex', 
    'Cortex', 
    adata_full_clean.obs[region_col]
)

unique_grouped_regions = sorted(adata_full_clean.obs['grouped_region'].unique().astype(str))

# 3. DYNAMIC COLOR PALETTE
# Goal: "Cortex" is Solid Red. Everything else is Faded.

fallback_palette = cm.tab20(np.linspace(0, 1, len(unique_grouped_regions)))
manual_overrides = {
    'Hippocampus': 'lightblue',
    'Cerebellum': 'lightgreen',
    'Unassigned': 'grey'
}

global_color_map = {}

for i, region in enumerate(unique_grouped_regions):
    if region == "Cortex":
        # Solid Red for the main group
        global_color_map[region] = mcolors.to_rgba('tab:red', alpha=1.0)
    else:
        # Determine base color for others
        if region in manual_overrides:
            base = manual_overrides[region]
        else:
            base = fallback_palette[i]
        
        # Apply Transparency (Fade)
        global_color_map[region] = mcolors.to_rgba(base, alpha=0.2)

# 4. PLOTTING
fig, axes = plt.subplots(2, 2, figsize=(16, 18))

rows_config = [
    (0, cortex_up, "G1: Stroke Cortex Enriched"),
   # (1, adata_full_clean, "Baseline (Unmanipulated)"),
    (1, cortex_down, "G2: Healthy Cortex Enriched")
]

cols_config = [
    (0, "Stroke_FMT"),
    (1, "Healthy_FMT")
]

# --- HELPER FUNCTION FOR LABELS ---
def get_clean_labels(counts, threshold=2.0):
    total = counts.sum()
    labels_list = []
    autopct_list = []
    
    for name, val in counts.items():
        pct = (val / total) * 100
        if pct >= threshold:
            labels_list.append(name)
            autopct_list.append(f"{pct:.1f}%")
        else:
            labels_list.append("") 
            autopct_list.append("")
            
    return labels_list, autopct_list

for row_idx, adata_obj, manip_name in rows_config:
    for col_idx, fmt_group in cols_config:
        
        ax = axes[row_idx, col_idx]
        
        # FIX: Access .obs for subsetting
        subset = adata_obj[adata_obj.obs[fmt_col] == fmt_group].copy()
        
        # Filter bad coords (using .obs)
        subset = subset[~subset.obs[region_col].astype(str).str.match(coordinate_pattern)]
        
        # APPLY GROUPING ON THE FLY (using .obs)
        # Using numpy where for vectorized conditional assignment
        subset.obs['grouped_region'] = np.where(
            subset.obs[napari_col] == 'Cortex', 
            'Cortex', 
            subset.obs[region_col]
        )
        
        # Count based on the NEW grouped column
        counts = subset.obs['grouped_region'].value_counts().sort_index()
        
        # Get Colors
        current_colors = [global_color_map.get(str(r), 'lightgrey') for r in counts.index]
        
        # Labels
        clean_labels, clean_autopct = get_clean_labels(counts, threshold=2.0)

        # Plot
        wedges, texts, autotexts = ax.pie(
            counts,
            labels=clean_labels, 
            autopct='%1.1f%%',   
            colors=current_colors,
            startangle=140,
            pctdistance=0.85,
            labeldistance=1.05,
            textprops={'fontsize': 10},
        )
        
        # Styling
        for i, t in enumerate(autotexts):
            t.set_text(clean_autopct[i])
            if clean_autopct[i] != "":
                t.set_fontsize(9)
                t.set_fontweight('bold')
                
                # Make "Cortex" text white (since background is dark red)
                if clean_labels[i] == "Cortex":
                    t.set_color('white')
                else:
                    t.set_color('black')

        # Add Donut Circle
        centre_circle = plt.Circle((0,0), 0.70, fc='white')
        ax.add_artist(centre_circle)
        
        ax.set_title(f"{manip_name}\n({fmt_group})", fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig(r"/path/to/output/Cortex_Enrichment_PieCharts.png", dpi=300, bbox_inches='tight')
plt.show()

In [50]:
plt.show()

In [ ]:
from typing import Dict, Union
import numpy as np
import pandas as pd
from anndata import AnnData
adata_full=adata_full[adata_full.obs['napari_region']!='Olfactory']
adata_full=adata_full[adata_full.obs['FMT']!='Cntrl']


random_state=0
def alloc_by_baseline_with_caps(baseline_counts: pd.Series, caps: pd.Series, total: int) -> pd.Series:
    """
    Allocate 'total' items across indices in proportion to 'baseline_counts',
    without exceeding 'caps' (remaining available per region).
    Simple proportional allocation + small fix if caps bind.
    """
    out = pd.Series(0, index=baseline_counts.index, dtype=int)
    remain = int(total)
    # weights from baseline (relative proportions)
    w = baseline_counts.clip(lower=0).astype(float)
    if w.sum() == 0 or remain <= 0:
        return out

    # 1st pass: proportional
    raw = w / w.sum() * remain
    base = np.floor(raw).astype(int)

    # respect caps
    take = base.clip(upper=caps.astype(int))
    out += take
    remain -= int(take.sum())

    # 2nd pass: give leftover to the largest fractional remainders, respecting caps
    if remain > 0:
        frac = (raw - base).sort_values(ascending=False)
        for idx in frac.index:
            if remain == 0:
                break
            if out[idx] < int(caps.get(idx, 0)):
                out[idx] += 1
                remain -= 1
    return out

def set_region_abundance_by_FMT(
    adata: AnnData,
    target_by_fmt: Dict[str, Dict[str, Union[int, float]]],
    total_per_fmt: Union[int, Dict[str, int], None] = None,
    region_col: str = "napari_region",
    fmt_col: str = "FMT",
    random_state: int | None =0,
) -> AnnData:
    """
    Select cells per FMT to match targets for specified regions (counts or fractions).
    All unspecified regions keep their *relative proportions* (baseline mix within that FMT),
    subject to availability caps. Ensures per-FMT totals are <= t and never exceed availability.
    """
    rng = np.random.default_rng(random_state)

    # Use existing labels only; avoid turning NaNs into "nan" strings
    obs_rf = adata.obs[[region_col, fmt_col]].dropna().copy()

    # availability table: counts per FMT x region
    avail = obs_rf.groupby([fmt_col, region_col]).size().unstack(fill_value=0)

    def _alloc_by_baseline_with_caps(baseline_counts: pd.Series,
                                     caps: pd.Series,
                                     total: int) -> pd.Series:
        """
        Allocate 'total' across indices proportionally to baseline_counts, without exceeding caps.
        Two-pass: floor proportional then distribute leftover by largest remainders.
        """
        out = pd.Series(0, index=baseline_counts.index, dtype=int)
        total = int(total)
        if total <= 0 or baseline_counts.sum() <= 0 or caps.sum() <= 0:
            return out

        weights = baseline_counts.clip(lower=0).astype(float)
        weights = weights / (weights.sum() if weights.sum() > 0 else 1.0)

        raw = weights * total
        base = np.floor(raw).astype(int)
        take = base.clip(upper=caps.astype(int))
        out += take
        rem = total - int(take.sum())
        if rem <= 0:
            return out

        frac = (raw - base).sort_values(ascending=False)
        for idx in frac.index:
            if rem == 0:
                break
            if out[idx] < int(caps.get(idx, 0)):
                out[idx] += 1
                rem -= 1
        return out

    selected = []

    for fmt in avail.index:
        row = avail.loc[fmt]                     # available counts per region for this FMT
        total_avail = int(row.sum())

        # choose per-FMT target total t
        if isinstance(total_per_fmt, dict):
            t = int(min(total_per_fmt.get(fmt, total_avail), total_avail))
        elif isinstance(total_per_fmt, int):
            t = int(min(total_per_fmt, total_avail))
        else:
            t = total_avail

        # baseline mix for unspecified regions (keep-as-is proportions)
        baseline = (row / row.sum()).fillna(0)

        desired = pd.Series(0, index=row.index, dtype=int)

        if fmt in target_by_fmt:
            tgt = target_by_fmt[fmt]

            # Determine if targets are explicit counts or fractions
            is_count = all(isinstance(v, (int, np.integer)) for v in tgt.values())

            if is_count:
                # 1) lock specified counts
                for r, v in tgt.items():
                    if r in desired.index:
                        desired[r] = int(v)
            else:
                # 1) lock specified fractions
                frac_map = {r: float(v) for r, v in tgt.items() if r in desired.index}
                sfrac = sum(frac_map.values())
                if sfrac > 1.0 and sfrac > 0:
                    frac_map = {r: v / sfrac for r, v in frac_map.items()}
                for r, fr in frac_map.items():
                    desired[r] = int(round(fr * t))

            # Cap specified to availability
            desired = desired.clip(upper=row)

            # 2) allocate leftover to unspecified regions by baseline ratios (respect caps)
            unspecified = [r for r in desired.index if r not in tgt]
            remain = max(0, t - int(desired.sum()))
            if unspecified and remain > 0:
                caps_unspec = (row - desired).loc[unspecified].clip(lower=0)
                alloc = _alloc_by_baseline_with_caps(baseline.loc[unspecified], caps_unspec, remain)
                desired.loc[unspecified] += alloc

        else:
            # No targets: keep baseline mix up to t (respect caps)
            caps_all = row.clip(lower=0)
            desired = _alloc_by_baseline_with_caps(row, caps_all, t)

        # Safety: never over t (small rounding trims)
        while int(desired.sum()) > t:
            biggest = desired.idxmax()
            if desired[biggest] == 0:
                break
            desired[biggest] -= 1

        # Final cap to availability
        desired = desired.clip(upper=row)

        # Sample the requested cells for this FMT
        sub = obs_rf[obs_rf[fmt_col] == fmt]
        for region, need in desired.items():
            n = int(need)
            if n <= 0:
                continue
            pool = sub.index[sub[region_col] == region].to_numpy()
            if n >= len(pool):
                selected.extend(pool.tolist())
            else:
                selected.extend(rng.choice(pool, size=n, replace=False).tolist())

    sel_set = set(selected)
    sel_index = [idx for idx in obs_rf.index if idx in sel_set]
    if not sel_index:
        raise ValueError("No cells selected.")

    return adata[sel_index].copy()

#()()()()()()()()()()()()()(()()()()())()()()()()()()()()()()()()()()()()()()()()()()()()()()()
#()()()()()()()()()()()()()(()()()()())()()()()()()()()()()()()()()()()()()()()()()()()()()()()


sample_col = "sample_ID"  # <--- CHANGE THIS to your actual sample column name
region_col = "napari_region"
# 1. Calculate the fraction of cells per region for each sample
# Counts per sample per region
counts_per_sample = adata_full.obs.groupby([sample_col, region_col]).size().unstack(fill_value=0)
# Convert to fractions (row-wise normalization)
fractions_per_sample = counts_per_sample.div(counts_per_sample.sum(axis=1), axis=0)

# 2. Calculate Mean and STD across samples
# Note: Using fractions directly (e.g. 0.25) instead of percent (25.0) to match target dict format
region_means = fractions_per_sample.mean()
region_stds = fractions_per_sample.std()

# 3. Define helper variables dynamically
# Dictionary to store stats for easy access if needed
stats = {}
regions_of_interest = ["Cortex", "Cerebellum", "Hippocampus"]

# Initialize variables with calculated values
# We clip the lower bound at 0.0 to avoid negative targets
cortex_n = region_means.get("Cortex", 0.0)
cortex_h = cortex_n + region_stds.get("Cortex", 0.0)
cortex_l = max(0.0, cortex_n - region_stds.get("Cortex", 0.0))

cerebellum_n = region_means.get("Cerebellum", 0.0)
cerebellum_h = cerebellum_n + region_stds.get("Cerebellum", 0.0)
cerebellum_l = max(0.0, cerebellum_n - region_stds.get("Cerebellum", 0.0))

hippocampus_n = region_means.get("Hippocampus", 0.0)
hippocampus_h = hippocampus_n + region_stds.get("Hippocampus", 0.0)
hippocampus_l = max(0.0, hippocampus_n - region_stds.get("Hippocampus", 0.0))



mean_cort=region_stds.get( "Cortex", 0.0)
print("--- Calculated Stats (Fractions) ---")
print(f"Cortex:      Mean={cortex_n:.4f}, STD={mean_cort:.4f}, High={cortex_h:.4f}, Low={cortex_l:.4f}")
print(f"Cerebellum:  Mean={cerebellum_n:.4f}, High={cerebellum_h:.4f}, Low={cerebellum_l:.4f}")
print(f"Hippocampus: Mean={hippocampus_n:.4f}, High={hippocampus_h:.4f}, Low={hippocampus_l:.4f}")

#()()()()()()()()()()()()()(()()()()())()()()()()()()()()()()()()()()()()()()()()()()()()()()()
# TARGET DEFINITIONS
#()()()()()()()()()()()()()(()()()()())()()()()()()()()()()()()()()()()()()()()()()()()()()()()

target_by_fmt_norm = {
    "Stroke_FMT": {"Cortex": cortex_n, "Hippocampus": hippocampus_n, "Cerebellum": cerebellum_n},
    "Healthy_FMT": {"Cortex": cortex_n, "Hippocampus": hippocampus_n, "Cerebellum": cerebellum_n},
    "Cntrl": {"Cortex": cortex_n, "Hippocampus": hippocampus_n, "Cerebellum": cerebellum_n},
}

# Note: In the original code, 'Cntrl' for Hippo_up was set to 'cortex_n'. 
# Assuming you want the Hippocampus normal baseline there, I changed it to 'hippocampus_n'.
target_by_fmt_Hippo_up = {
    "Stroke_FMT": {"Hippocampus": hippocampus_h},  
    "Healthy_FMT": {"Hippocampus": hippocampus_l},
    "Cntrl": {"Hippocampus": hippocampus_n},   
}
target_by_fmt_Hippo_down= {
    "Stroke_FMT": {"Hippocampus": hippocampus_l},   
    "Healthy_FMT": {"Hippocampus": hippocampus_h},
    "Cntrl": {"Hippocampus": hippocampus_n},    
}

target_by_Cortex_up = {
    "Stroke_FMT": {"Cortex": cortex_h},  
    "Healthy_FMT": {"Cortex": cortex_l},
    "Cntrl": {"Cortex": cortex_n},   
}
target_by_Cortex_down = {
    "Stroke_FMT": {"Cortex": cortex_l},   
    "Healthy_FMT": {"Cortex": cortex_h},
    "Cntrl": {"Cortex": cortex_n}, 
}

target_by_Cerebellum_up = {
    "Stroke_FMT": {"Cerebellum": cerebellum_h},   
    "Healthy_FMT": {"Cerebellum": cerebellum_l},
    "Cntrl": {"Cerebellum": cerebellum_n}, 
}
target_by_Cerebellum_down = {
    "Stroke_FMT": {"Cerebellum": cerebellum_l},   
    "Healthy_FMT": {"Cerebellum": cerebellum_h},
    "Cntrl": {"Cerebellum": cerebellum_n},    
}



#EXCLUDING BRAINSTEM
adata_full=adata_full[adata_full.obs['napari_region']!='Olfactory']



print(cortex_h)
print(cortex_n)
print(cortex_l)
# Run Sampling
#norm = set_region_abundance_by_FMT(adata_full, target_by_fmt_norm, total_per_fmt=50000, random_state=0)
#hippo_up = set_region_abundance_by_FMT(adata_full, target_by_fmt_Hippo_up, total_per_fmt=10000, random_state=0)
#hippo_down = set_region_abundance_by_FMT(adata_full, target_by_fmt_Hippo_down, total_per_fmt=10000, random_state=0)
cortex_up = set_region_abundance_by_FMT(adata_full, target_by_Cortex_up, total_per_fmt=125000, random_state=0)
cortex_down = set_region_abundance_by_FMT(adata_full, target_by_Cortex_down, total_per_fmt=125000, random_state=0)
#cerebellum_up = set_region_abundance_by_FMT(adata_full, target_by_Cerebellum_up, total_per_fmt=10000, random_state=0)
#cerebellum_down = set_region_abundance_by_FMT(adata_full, target_by_Cerebellum_down, total_per_fmt=10000, random_state=0)


In [ ]:
# Check GLOBAL percentages (Weighted), not Mean of Percents
print(adata_full[adata_full.obs['FMT']=='Stroke_FMT'].obs['napari_region'].value_counts(normalize=True)['Cortex'])
print(adata_full[adata_full.obs['FMT']=='Healthy_FMT'].obs['napari_region'].value_counts(normalize=True)['Cortex'])

print("--- Global Cortex % (Stroke) ---")
print(cortex_down[cortex_down.obs['FMT']=='Stroke_FMT'].obs['napari_region'].value_counts(normalize=True)['Cortex'])
print(cortex_down[cortex_down.obs['FMT']=='Healthy_FMT'].obs['napari_region'].value_counts(normalize=True)['Cortex'])

print("--- Global Cortex % (Healthy) ---")
print(cortex_up[cortex_up.obs['FMT']=='Healthy_FMT'].obs['napari_region'].value_counts(normalize=True)['Cortex'])
print(cortex_up[cortex_up.obs['FMT']=='Stroke_FMT'].obs['napari_region'].value_counts(normalize=True)['Cortex'])


In [ ]:
# Check GLOBAL percentages (Weighted), not Mean of Percents
print(adata_full.obs['broad_quint_region'].value_counts(normalize=True)['Cortex'])

print("--- Global Cortex % (Stroke) ---")
print(cortex_down_q[cortex_down_q.obs['FMT']=='Stroke_FMT'].obs['broad_quint_region'].value_counts(normalize=True)['Cortex'])
print(cortex_down_q[cortex_down_q.obs['FMT']=='Healthy_FMT'].obs['broad_quint_region'].value_counts(normalize=True)['Cortex'])

print("--- Global Cortex % (Healthy) ---")
print(cortex_up_q[cortex_up_q.obs['FMT']=='Healthy_FMT'].obs['broad_quint_region'].value_counts(normalize=True)['Cortex'])
print(cortex_up_q[cortex_up_q.obs['FMT']=='Stroke_FMT'].obs['broad_quint_region'].value_counts(normalize=True)['Cortex'])


In [ ]:

print('2')
cortex_up_genes=ezy.rank_DE(
    cortex_up,
    groupby="FMT",
    comparisons=[['Stroke_FMT','Healthy_FMT']],
    #extra_filter={"cell_type": ["Astro","Micro.PVM"]}  # optional extra filters
)
print('3')
baseline=ezy.rank_DE(
    adata_full,
    groupby="FMT",
    comparisons=[['Stroke_FMT','Healthy_FMT']],
    #extra_filter={"cell_type": ["Astro","Micro.PVM"]}  # optional extra filters
)
print('4')
cortex_down_genes=ezy.rank_DE(
    cortex_down,
    groupby="FMT",
    comparisons=[['Stroke_FMT','Healthy_FMT']],
    #extra_filter={"cell_type": ["Astro","Micro.PVM"]}  # optional extra filters
)

cortex_up_genes_A=ezy.rank_DE(
    cortex_up,
    groupby="FMT",
    comparisons=[['Stroke_FMT','Healthy_FMT']],
    extra_filter={"cell_type": ["Astro"]}  # optional extra filters
)
print('3')
baseline_A=ezy.rank_DE(
    adata_full,
    groupby="FMT",
    comparisons=[['Stroke_FMT','Healthy_FMT']],
    extra_filter={"cell_type": ["Astro"]}  # optional extra filters
)
print('4')
cortex_down_genes_A=ezy.rank_DE(
    cortex_down_q, 
    groupby="FMT",
    comparisons=[['Stroke_FMT','Healthy_FMT']],
    extra_filter={"cell_type": ["Astro"]}  # optional extra filters
)
#########################################################
cortex_up_genes_M=ezy.rank_DE(
    cortex_up,
    groupby="FMT",
    comparisons=[['Stroke_FMT','Healthy_FMT']],
    extra_filter={"cell_type": ["Micro.PVM"]}  # optional extra filters
)
print('3')
baseline_M=ezy.rank_DE(
    adata_full,
    groupby="FMT",
    comparisons=[['Stroke_FMT','Healthy_FMT']],
    extra_filter={"cell_type": ["Micro.PVM"]}  # optional extra filters
)
print('4')
cortex_down_genes_M=ezy.rank_DE(
    cortex_down, 
    groupby="FMT",
    comparisons=[['Stroke_FMT','Healthy_FMT']],
    extra_filter={"cell_type": ["Micro.PVM"]}  # optional extra filters
)



In [ ]:
list(adata_full.obs['cell_type'].unique())

In [ ]:
log_thresh=0.2
import matplotlib.pyplot as plt
gene_num = 40  # how many labels to show on volcano plots
top_n_for_deltas = 30  # how many top |score| genes to use when computing deltas
def plot_celltype_counts_per_FMT(adata, top_n,colors=[(249/255,100/255,149/255),(85/255,160/255,251/255),'green']):
    ds = adata.copy()
    counts = ds.obs.groupby(["FMT", "cell_type"]).size()
    top_clusters = (
        counts
        .groupby(level="cell_type")
        .sum()
        .nlargest(top_n)
        .index
    )
    df = counts.unstack(level="FMT").loc[top_clusters].fillna(0)
    df = df[["Stroke_FMT", "Healthy_FMT"]]
    print(df.sum(axis=0))
    df_percent = df.div(df.sum(axis=0), axis=1) * 100
    df_percent = df_percent.rename(columns={"Stroke_FMT": "Stroke-FMT", "Healthy_FMT": "Healthy-FMT"})

    ax = df_percent.plot(
        kind="bar",
        color={"Stroke-FMT": colors[0], "Healthy-FMT": colors[1],"Cntrl":colors[2]},
        figsize=(8,4),
        width=0.8,
    )




    # after ax = df.plot(...)
   
    ax.legend( fontsize=12, title_fontsize=15, loc="upper right")
    ax.set_ylabel("Percentage of Cell Type", fontsize=12)
    print(f"Cell Cluster Abundance: Stroke-FMT vs. Healthy-FMT")
    ax.tick_params(axis='x', labelsize=12)
    ax.tick_params(axis='y', labelsize=12)

    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()



# ---------- helper to get symmetric-difference "deltas" between up and down ----------

def get_deltas(up_df, down_df, n=30):
    """Return symmetric difference of top-n genes by |score| between up and down."""
    k_up = n // 2
    k_dn = n - k_up

    # Top up-regulated (largest positive scores)
    up_up   = up_df.sort_values('score', ascending=False).head(k_up)['gene']
    down_up = down_df.sort_values('score', ascending=False).head(k_up)['gene']

    # Top down-regulated (most negative scores)
    up_dn   = up_df.sort_values('score', ascending=True).head(k_dn)['gene']
    down_dn = down_df.sort_values('score', ascending=True).head(k_dn)['gene']

    deltas_up   = list(set(up_up) ^ set(down_up))
    deltas_down = list(set(up_dn) ^ set(down_dn))
    return deltas_up + deltas_down

def get_logfc_genes(up_df, down_df, threshold):
    """
    Return unique list of genes with log2fc >= threshold from both dataframes.
    
    Parameters:
    up_df, down_df: DataFrames containing 'gene' and 'log2fc' columns.
    threshold: The minimum log2 fold-change required to include the gene.
    """
    merged = pd.merge(up_df, down_df, on='gene', suffixes=('_G1', '_G2'))
    
    # Calculate the absolute difference between the two log2fc columns
    merged['diff'] = (merged['log2fc_G1'] - merged['log2fc_G2']).abs()
    
    # Filter for genes where the difference meets the threshold
    delta_genes = merged[merged['diff'] >= threshold]['gene'].tolist()
    
    return delta_genes
def get_logfc_data(up_df, down_df, threshold):
    """
    Return unique list of genes with log2fc >= threshold from both dataframes.
    
    Parameters:
    up_df, down_df: DataFrames containing 'gene' and 'log2fc' columns.
    threshold: The minimum log2 fold-change required to include the gene.
    """
    merged = pd.merge(up_df, down_df, on='gene', suffixes=('_G1', '_G2'))
    
    # Calculate the absolute difference between the two log2fc columns
    merged['diff'] = (merged['log2fc_G1'] - merged['log2fc_G2']).abs()
    
    # Filter for genes where the difference meets the threshold
    delta_genes = merged[merged['diff'] >= threshold]
    
    return delta_genes

import pandas as pd

def get_opposite_genes(df1, df2, threshold=0):
    """
    Return genes that have opposite log2fc signs between two dataframes 
    and a total difference >= threshold.
    
    Parameters:
    df1, df2: DataFrames containing 'gene' and 'log2fc' columns.
    threshold: The minimum absolute difference required between the two values.
    """
    # Merge the two dataframes
    merged = pd.merge(df1, df2, on='gene', suffixes=('_G1', '_G2'))
    
    # 1. Identify opposite signs 
    # (Positive * Negative = Negative)
    opposite_mask = (merged['log2fc_G1'] * merged['log2fc_G2']) < 0
    
    # 2. Calculate the absolute difference (total magnitude of the flip)
    merged['diff'] = (merged['log2fc_G1'] - merged['log2fc_G2']).abs()
    
    # 3. Filter: Must have opposite signs AND meet the difference threshold
    final_genes = merged[opposite_mask & (merged['diff'])]
    
    # Sort by the biggest difference for easier reading
    return final_genes.sort_values(by='diff', ascending=False)

# =========================
# 1) HIPPOCAMPUS: Up vs Down
# =========================

# 2) CORTEX: Up vs Down
# ======================

# Volcano plots
ezy.plot.volcano(
    cortex_up_genes,
    y_axis='score',
    default_colors=['grey', 'grey'],
    significant_colors=[(249/255,100/255,149/255),(85/255,160/255,251/255)],
    balance_labels=True,
    return_highlights=False,
    top_labels=gene_num,
    figsize=(6.1,6.6),
    sig_markers=["+", "D"],
    title='Whole Brain – G1'
)
ezy.plot.volcano(
    baseline,
    y_axis='score',
    default_colors=['grey', 'grey'],

    significant_colors=['black','black'],
    balance_labels=True,
    return_highlights=False,
    top_labels=gene_num,
    figsize=(6.1,6.6),
    sig_markers=["+", "D"],
    title='Whole Brain - No Adjustment'
)

ezy.plot.volcano(
    cortex_down_genes,
    y_axis='score',
    default_colors=['grey', 'grey'],
    #significant_colors=[(249/255,100/255,149/255),(85/255,160/255,251/255)],
    significant_colors=['gold', 'magenta'],
    balance_labels=True,
    return_highlights=False,
    top_labels=gene_num,
    figsize=(6.1,6.6),
    sig_markers=["+", "D"],
    title='Whole Brain – G2'
)

# Deltas for bars
cortex_deltas = get_deltas(cortex_up_genes, cortex_down_genes, n=top_n_for_deltas)
cortex_stars = get_logfc_genes(cortex_up_genes, cortex_down_genes, threshold=log_thresh)
cortex_data = get_logfc_data(cortex_up_genes, cortex_down_genes, threshold=log_thresh)
cortex_opp_genes=get_opposite_genes(cortex_up_genes,cortex_down_genes)
print(cortex_stars)
# Bar plots
ezy.plot.feats_bar(
    cortex_up_genes,
    orientation='horizontal',
    bold_genes=cortex_deltas,
    starred_genes=cortex_stars,
    bold_colors=['blue','red'],
    fig_size=(6.1,6.6),
    num_feats=30,
    title='Whole Brain – G1'
)
ezy.plot.feats_bar(
    baseline,
    orientation='horizontal',
    #bold_genes=cortex_deltas,
    colors=[ "#C6C6C6FF","#707070FF"],
    bold_colors=[ "#5F5F5FFF","#292929FF"],
    bold_genes=cortex_deltas,
    starred_genes=cortex_stars,
    fig_size=(6.1,6.6),
    num_feats=30,
    title='Whole Brain – No Adjustment'
)

ezy.plot.feats_bar(
    cortex_down_genes,
    orientation='horizontal',
    bold_genes=cortex_deltas,
    starred_genes=cortex_stars,

    colors=[ "#FF64FCFF",'gold'],
    bold_colors=["#A500A2FF","#E39400FF"],

    fig_size=(6.1,6.6),
    num_feats=30,
    title='Whole Brain – G2'
)

#plot_celltype_counts_per_FMT(cortex_up, 16)
#plot_celltype_counts_per_FMT(cortex_down, 16,colors=['gold', 'magenta','green'])


In [41]:
cortex_opp_genes[['gene',
'log2fc_G1',
'log2fc_G2','diff','score_G1','score_G2']].sort_values(by='diff', ascending=False).to_csv(r"/path/to/output/lfc_inversion_q.csv")

cortex_data[['gene',
'log2fc_G1',
'log2fc_G2','diff','score_G1','score_G2']].sort_values(by='diff', ascending=False).to_csv(r"/path/to/output/delta_lfc_thresh_q.csv")


In [ ]:
#Astrocyte Manipulated DE
ezy.plot.volcano(
    cortex_up_genes_A,
    y_axis='score',
    default_colors=['grey', 'grey'],
    significant_colors=[(249/255,100/255,149/255),(85/255,160/255,251/255)],
    balance_labels=True,
    return_highlights=False,
    top_labels=gene_num,
    figsize=(6.1, 6.6),
    sig_markers=["+", "D"],
    title='Astrocytes – G1'
)

ezy.plot.volcano(
    baseline_A,
    y_axis='score',
    default_colors=['grey', 'grey'],

    significant_colors=['black','black'],
    balance_labels=True,
    return_highlights=False,
    top_labels=gene_num,
    figsize=(6.1, 6.6),
    sig_markers=["+", "D"],
    title='Astrocytes – No Adjustment'
)

ezy.plot.volcano(
    cortex_down_genes_A,
    y_axis='score',
    default_colors=['grey', 'grey'],
    #significant_colors=[(249/255,100/255,149/255),(85/255,160/255,251/255)],
    significant_colors=['gold', 'magenta'],
    balance_labels=True,
    return_highlights=False,
    top_labels=gene_num,
    figsize=(6.1, 6.6),
    sig_markers=["+", "D"],
    title='Astrocytes – G2'
)

# Deltas for bars
cortex_deltas = get_deltas(cortex_up_genes_A, cortex_down_genes_A, n=top_n_for_deltas)
cortex_stars = get_logfc_genes(cortex_up_genes_A, cortex_down_genes_A, threshold=log_thresh)
print(cortex_stars)
# Bar plots
ezy.plot.feats_bar(
    cortex_up_genes_A,
    orientation='horizontal',
    bold_genes=cortex_deltas,
    starred_genes=cortex_stars,
    bold_colors=['blue','red'],
    fig_size=(6.1,6.6),
    num_feats=30,
    title='Astrocytes – G1'
)
ezy.plot.feats_bar(
    baseline_A,
    orientation='horizontal',
    #bold_genes=cortex_deltas,
    colors=[ "#C6C6C6FF","#707070FF"],
    bold_colors=[ "#5F5F5FFF","#292929FF"],
    bold_genes=cortex_deltas,
    starred_genes=cortex_stars,
    fig_size=(6.1,6.6),
    num_feats=30,
    title='Astrocytes – No Adjustment'
)

ezy.plot.feats_bar(
    cortex_down_genes_A,
    orientation='horizontal',
    bold_genes=cortex_deltas,
    starred_genes=cortex_stars,

    colors=[ "#FF64FCFF",'gold'],
    bold_colors=["#A500A2FF","#E39400FF"],

    fig_size=(6.1,6.6),
    num_feats=30,
    title='Astrocytes – G2'
)


In [ ]:
ezy.plot.volcano(
    cortex_up_genes_M,
    y_axis='score',
    default_colors=['grey', 'grey'],
    significant_colors=[(249/255,100/255,149/255),(85/255,160/255,251/255)],
    balance_labels=True,
    return_highlights=False,
    top_labels=gene_num,
    figsize=(6.1, 6.6),
    sig_markers=["+", "D"],
    title='Micro.PVM – G1'
)

ezy.plot.volcano(
    baseline_M,
    y_axis='score',
    default_colors=['grey', 'grey'],

    significant_colors=['black','black'],
    balance_labels=True,
    return_highlights=False,
    top_labels=gene_num,
    figsize=(6.1, 6.6),
    sig_markers=["+", "D"],
    title='Micro.PVM – No Adjustment'
)

ezy.plot.volcano(
    cortex_down_genes_M,
    y_axis='score',
    default_colors=['grey', 'grey'],
    #significant_colors=[(249/255,100/255,149/255),(85/255,160/255,251/255)],
    significant_colors=['gold', 'magenta'],
    balance_labels=True,
    return_highlights=False,
    top_labels=gene_num,
    figsize=(6.1, 6.6),
    sig_markers=["+", "D"],
    title='Micro.PVM – G2'
)

# Deltas for bars
cortex_deltas = get_deltas(cortex_up_genes_M, cortex_down_genes_M, n=top_n_for_deltas)
cortex_stars = get_logfc_genes(cortex_up_genes_M, cortex_down_genes_M, threshold=log_thresh)
cortex_data_M = get_logfc_data(cortex_up_genes_M, cortex_down_genes_M, threshold=log_thresh)
cortex_opp_genes_M=get_opposite_genes(cortex_up_genes_M,cortex_down_genes_M)

print(cortex_stars)
# Bar plots
ezy.plot.feats_bar(
    cortex_up_genes_M,
    orientation='horizontal',
    bold_genes=cortex_deltas,
    starred_genes=cortex_stars,
    bold_colors=['blue','red'],
    fig_size=(6.1,6.6),
    num_feats=30,
    title='Micro.PVM – G1'
)
ezy.plot.feats_bar(
    baseline_M,
    orientation='horizontal',
    #bold_genes=cortex_deltas,
    colors=[ "#C6C6C6FF","#707070FF"],
    bold_colors=[ "#5F5F5FFF","#292929FF"],
    bold_genes=cortex_deltas,
    starred_genes=cortex_stars,
    fig_size=(6.1,6.6),
    num_feats=30,
    title='Micro.PVM – No Adjustment'
)

ezy.plot.feats_bar(
    cortex_down_genes_M,
    orientation='horizontal',
    bold_genes=cortex_deltas,
    starred_genes=cortex_stars,

    colors=[ "#FF64FCFF",'gold'],
    bold_colors=["#A500A2FF","#E39400FF"],

    fig_size=(6.1,6.6),
    num_feats=30,
    title='Micro.PVM – G2'
)


In [12]:
cortex_opp_genes_M[['gene',
'log2fc_G1',
'log2fc_G2','diff','score_G1','score_G2']].sort_values(by='diff', ascending=False).to_csv(r"/path/to/output/lfc_inversion_M.csv")

cortex_data_M[['gene',
'log2fc_G1',
'log2fc_G2','diff','score_G1','score_G2']].sort_values(by='diff', ascending=False).to_csv(r"/path/to/output/delta_lfc_thresh_M.csv")


In [ ]:
def plot_celltype_counts_per_FMT_trip(
    adata1,
    adata2,
    adata3, 
    top_n,
    label1="Up",
    label2="Down",
    label3="Base",
    fmt_cols=("Stroke_FMT", "Healthy_FMT"),
    colors=None,
    hatches=None,
    gap=0.1, 
    annotate_significance=True,
    alpha=0.05,
    fig_size=(14,10)
):
    """
    Plot 6 bars per cell type with significance testing for 1vs2, 2vs3, and 1vs3.
    """

    ds1 = adata1.copy()
    ds2 = adata2.copy()
    ds3 = adata3.copy()

    # 1. Raw counts
    counts1 = ds1.obs.groupby(["FMT", "cell_type"]).size()
    counts2 = ds2.obs.groupby(["FMT", "cell_type"]).size()
    counts3 = ds3.obs.groupby(["FMT", "cell_type"]).size()

    # 2. Totals
    total1_by_fmt = counts1.groupby(level="FMT").sum()
    total2_by_fmt = counts2.groupby(level="FMT").sum()
    total3_by_fmt = counts3.groupby(level="FMT").sum()

    # 3. Top N selection
    combined_counts = (
        counts1.add(counts2, fill_value=0).add(counts3, fill_value=0)
        .groupby(level="cell_type").sum()
    )
    top_clusters = combined_counts.nlargest(top_n).index

    # 4. Percent calc
    def _percent_df(counts):
        df = counts.unstack(level="FMT").fillna(0)
        for c in fmt_cols:
            if c not in df.columns:
                df[c] = 0.0
        df = df.reindex(index=top_clusters, columns=fmt_cols, fill_value=0)
        return df.div(df.sum(axis=0), axis=1) * 100

    df1 = _percent_df(counts1)
    df2 = _percent_df(counts2)
    df3 = _percent_df(counts3)

    # 5. Combined DataFrame
    col_names = [
        f"{label1}-Stroke", f"{label1}-Healthy",
        f"{label2}-Stroke", f"{label2}-Healthy",
        f"{label3}-Stroke", f"{label3}-Healthy",
    ]

    combined = pd.DataFrame({
        col_names[0]: df1[fmt_cols[0]],
        col_names[1]: df1[fmt_cols[1]],
        col_names[2]: df2[fmt_cols[0]],
        col_names[3]: df2[fmt_cols[1]],
        col_names[4]: df3[fmt_cols[0]],
        col_names[5]: df3[fmt_cols[1]],
    }, index=top_clusters)

    # 6. Setup Plot
    if colors is None:
        colors = [
            (249/255,100/255,149/255), (85/255,160/255,251/255), 
            "gold", "magenta", 
            "gray", "silver"
        ]
    if hatches is None:
        hatches = [""] * 6

    n = len(top_clusters)
    x = np.arange(n)
    bar_width = 0.12 
    
    # X Positions: G1(S,H) ... G2(S,H) ... G3(S,H)
    # Center is G2. 
    x_g2_s = x - (bar_width / 2)
    x_g2_h = x + (bar_width / 2)
    
    x_g1_h = x_g2_s - gap - (bar_width / 2)
    x_g1_s = x_g1_h - bar_width
    
    x_g3_s = x_g2_h + gap + (bar_width / 2)
    x_g3_h = x_g3_s + bar_width
    
    # List of X coordinates for easy indexing later: 0=G1S, 1=G1H, 2=G2S, 3=G2H, 4=G3S, 5=G3H
    x_coords_arrays = [x_g1_s, x_g1_h, x_g2_s, x_g2_h, x_g3_s, x_g3_h]

    fig, ax = plt.subplots(figsize=fig_size)

    # Draw Bars
    bars = []
    for k in range(6):
        b = ax.bar(x_coords_arrays[k], combined[col_names[k]], 
                   width=bar_width, color=colors[k], hatch=hatches[k], 
                   label=col_names[k], edgecolor='#4A4A4A')
        bars.append(b)
    
    # Unpack for clarity (optional, but helpful for debugging)
    b1_s, b1_h, b2_s, b2_h, b3_s, b3_h = bars

    ax.set_xticks(x)
    ax.set_xticklabels(top_clusters, rotation=45, ha="right", fontsize=12)
    ax.set_ylabel("Percentage of Cell Type", fontsize=12)
    ax.legend(fontsize=10, title="Group x FMT", title_fontsize=12)
    ax.tick_params(axis='y', labelsize=12)

    # 7. Significance Logic (Updated for 3-way comparisons)
    if annotate_significance:
        global_max_y = combined.values.max()
        vertical_step = 0.08 * global_max_y
        max_y_plotted = global_max_y

        def _p_to_stars(p):
            if p < 0.001: return "***"
            elif p < 0.01: return "**"
            elif p < 0.05: return "*"
            else: return ""

        # Define comparisons: (Label, DataA, TotalA, DataB, TotalB, GroupIdxA, GroupIdxB)
        # Group Indices: 1, 2, 3
        comparisons_config = [
            # Short brackets first (Pyramid stacking)
            ('1vs2', counts1, total1_by_fmt, counts2, total2_by_fmt, 1, 2),
            ('2vs3', counts2, total2_by_fmt, counts3, total3_by_fmt, 2, 3),
            # Long bracket last
            ('1vs3', counts1, total1_by_fmt, counts3, total3_by_fmt, 1, 3),
        ]

        for i, ct in enumerate(top_clusters):
            last_bracket_y = 0 
            
            for (comp_name, c_a, t_a, c_b, t_b, g_idx_a, g_idx_b) in comparisons_config:
                for j, fmt in enumerate(fmt_cols):
                    # j=0 Stroke, j=1 Healthy
                    
                    # Fisher Test
                    cnt_a = c_a.get((fmt, ct), 0)
                    cnt_b = c_b.get((fmt, ct), 0)
                    tot_a = t_a.get(fmt, 0)
                    tot_b = t_b.get(fmt, 0)
                    
                    if tot_a == 0 or tot_b == 0: continue
                    
                    try:
                        _, pval = fisher_exact([[cnt_a, tot_a - cnt_a], [cnt_b, tot_b - cnt_b]])
                    except:
                        continue
                        
                    if pval < alpha:
                        stars = _p_to_stars(pval)
                        
                        # Determine indices in the 0..5 bar list
                        # G1: S=0, H=1 | G2: S=2, H=3 | G3: S=4, H=5
                        # Formula: idx = (GroupNum-1)*2 + (0 if Stroke else 1)
                        
                        idx_a = (g_idx_a - 1) * 2 + j
                        idx_b = (g_idx_b - 1) * 2 + j
                        
                        # Indices involved in the span (inclusive)
                        start_idx = min(idx_a, idx_b)
                        end_idx = max(idx_a, idx_b)
                        
                        # 1. Calculate max bar height in this span
                        # We check heights of all bars physically between start and end
                        span_heights = []
                        for k in range(start_idx, end_idx + 1):
                            span_heights.append(bars[k][i].get_height())
                        
                        max_bar_h = max(span_heights) if span_heights else 0
                        
                        # 2. Clearance logic
                        clearance = max_bar_h + (0.02 * global_max_y)
                        
                        # 3. Stack on top of previous brackets for this cluster
                        if clearance <= last_bracket_y:
                            y = last_bracket_y + vertical_step
                        else:
                            y = clearance
                        
                        # Update tracker
                        last_bracket_y = y
                        max_y_plotted = max(max_y_plotted, y)
                        
                        # Draw
                        x_left = x_coords_arrays[start_idx][i]
                        x_right = x_coords_arrays[end_idx][i]
                        
                        ax.plot([x_left, x_left, x_right, x_right],
                                [y - 0.01*global_max_y, y, y, y - 0.01*global_max_y],
                                color='black', linewidth=1.0)
                        
                        # Star text
                        ax.text((x_left + x_right) / 2, y + 0.01*global_max_y, stars,
                                ha="center", va="bottom", fontsize=10, color='black')

        ax.set_ylim(0, max_y_plotted * 1.15)

    plt.tight_layout()
    return ax
def plot_celltype_counts_per_FMT_pair(
    adata1,
    adata2,
    top_n,
    label1="Up",
    label2="Down",
    fmt_cols=("Stroke_FMT", "Healthy_FMT"),
    colors=None,
    hatches=None,
    gap=0.25,
    annotate_significance=True,
    alpha=0.05
):
    """
    Plot 4 bars per cell type with a gap between Up and Down.
    Optionally compute significance (Fisher's exact test) comparing
    Up vs Down within each FMT for each cell type.
    """

    ds1 = adata1.copy()
    ds2 = adata2.copy()

    # raw counts per (FMT, cell_type)
    counts1 = ds1.obs.groupby(["FMT", "cell_type"]).size()
    counts2 = ds2.obs.groupby(["FMT", "cell_type"]).size()

    # total counts per FMT (for "other cell types" in contingency table)
    total1_by_fmt = counts1.groupby(level="FMT").sum()
    total2_by_fmt = counts2.groupby(level="FMT").sum()

    # choose shared top_n cell types
    combined_counts = (
        counts1.add(counts2, fill_value=0)
        .groupby(level="cell_type").sum()
    )
    top_clusters = combined_counts.nlargest(top_n).index

    # helper: convert to % within each FMT (same as before)
    def _percent_df(counts):
        df = counts.unstack(level="FMT").fillna(0)
        df = df.reindex(index=top_clusters, columns=fmt_cols, fill_value=0)
        return df.div(df.sum(axis=0), axis=1) * 100

    df1 = _percent_df(counts1)
    df2 = _percent_df(counts2)

    # combined % table
    col_names = [
        f"{label1}-Stroke", f"{label1}-Healthy",
        f"{label2}-Stroke", f"{label2}-Healthy",
    ]

    combined = pd.DataFrame({
        col_names[0]: df1[fmt_cols[0]],
        col_names[1]: df1[fmt_cols[1]],
        col_names[2]: df2[fmt_cols[0]],
        col_names[3]: df2[fmt_cols[1]],
    }, index=top_clusters)

    # colors
    if colors is None:
        colors = [
            (249/255,100/255,149/255),
            (85/255,160/255,251/255),
            "gold",
            "magenta"
        ]

    # hatches
    if hatches is None:
        hatches = ["", "", "", ""]

    # bar positions
    n = len(top_clusters)
    x = np.arange(n)
    bar_width = 0.18

    x_up_healthy  = x - bar_width - gap/2
    x_up_stroke = x - gap/2
    x_dn_healthy  = x + gap/2
    x_dn_stroke = x + bar_width + gap/2

    fig, ax = plt.subplots(figsize=(12, 5))

    b_up_stroke = ax.bar(
        x_up_stroke,
        combined[col_names[0]],
        width=bar_width,
        color=colors[0],
        hatch=hatches[0],
        label=col_names[0],
        edgecolor='#4A4A4A'
    )
    b_up_healthy = ax.bar(
        x_up_healthy,
        combined[col_names[1]],
        width=bar_width,
        color=colors[1],
        hatch=hatches[1],
        label=col_names[1],
        edgecolor='#4A4A4A'
    )
    b_dn_stroke = ax.bar(
        x_dn_stroke,
        combined[col_names[2]],
        width=bar_width,
        color=colors[2],
        hatch=hatches[2],
        label=col_names[2],
        edgecolor='#4A4A4A'
    )
    b_dn_healthy = ax.bar(
        x_dn_healthy,
        combined[col_names[3]],
        width=bar_width,
        color=colors[3],
        hatch=hatches[3],
        label=col_names[3],
        edgecolor='#4A4A4A'
    )

    # axis labels
    ax.set_xticks(x)
    ax.set_xticklabels(top_clusters, rotation=45, ha="right", fontsize=12)
    ax.set_ylabel("Percentage of Cell Type", fontsize=12)
    ax.legend(fontsize=10, title="Group × FMT", title_fontsize=12)
    ax.tick_params(axis='y', labelsize=12)

    # ---------- Significance: Up vs Down within each FMT ----------
    if annotate_significance:
        ymax = combined.values.max()

        def _p_to_stars(p):
            if p < 0.001:
                return "***"
            elif p < 0.01:
                return "**"
            elif p < 0.05:
                return "*"
            else:
                return ""

        for i, ct in enumerate(top_clusters):
            # for each FMT: Stroke_FMT (index 0), Healthy_FMT (index 1)
            for j, fmt in enumerate(fmt_cols):
                # counts for this cell type and FMT in each dataset
                c1 = counts1.get((fmt, ct), 0)
                c2 = counts2.get((fmt, ct), 0)

                # total cells of that FMT in each dataset
                t1 = total1_by_fmt.get(fmt, 0)
                t2 = total2_by_fmt.get(fmt, 0)

                # if any zero totals, skip
                if t1 == 0 or t2 == 0:
                    continue

                # contingency:
                # [ [Up: this cell type, Up: others],
                #   [Down: this cell type, Down: others] ]
                table = [
                    [c1, t1 - c1],
                    [c2, t2 - c2]
                ]

                try:
                    _, pval = fisher_exact(table)
                except Exception:
                    continue

                if pval < alpha:
                    stars = _p_to_stars(pval)

                    # y position slightly above the taller of the two bars
                    if j == 0:  # Stroke
                        y1 = max(b_up_stroke[i].get_height(),b_up_healthy[i].get_height())
                        y2 = max(b_dn_stroke[i].get_height(),b_dn_healthy[i].get_height())
                        x1 = x_up_stroke[i]
                        x2 = x_dn_stroke[i]
                        offset = 0.03 * ymax
                        color_='black'
                    else:       # Healthy
                        y1 = max(b_up_stroke[i].get_height(),b_up_healthy[i].get_height())
                        y2 = max(b_dn_stroke[i].get_height(),b_dn_healthy[i].get_height())
                        x1 = x_up_healthy[i]
                        x2 = x_dn_healthy[i]
                        offset = 0.1 * ymax  # a bit higher to avoid overlap
                        color_='black'

                    y = max(y1, y2) + offset

                    # draw bracket
                    ax.plot([x1, x1, x2, x2],
                            [y - 0.01*ymax, y, y, y - 0.01*ymax],
                            color=color_, linewidth=1.0)

                    # add stars
                    ax.text(
                        (x1 + x2) / 2,
                        y + 0.01*ymax,
                        stars,
                        ha="center",
                        va="bottom",
                        fontsize=10,
                        color=color_
                    )

    plt.tight_layout()
    return ax
'''plot_celltype_counts_per_FMT_pair(
    cortex_up,
    cortex_down,
    top_n=16,
    label1="G1",
    label2="G2",
    hatches=["","","",""],
    colors=[
        (249/255, 100/255, 149/255),  # Up Stroke
        (85/255, 160/255, 251/255),   # Up Healthy
        #(249/255, 100/255, 149/255),  # Up Stroke
        #(85/255, 160/255, 251/255),  
        'gold',  # Up Stroke
        'magenta'
        ]
)'''
plot_celltype_counts_per_FMT_trip(
     cortex_up, 
     adata_full,
     cortex_down, 
    fig_size=[14,5],
     top_n=10,
     label1="G1", 
     label2="Base", 
     label3="G2",
     annotate_significance=False,
     colors=[
        (249/255, 100/255, 149/255), # G1 S
        (85/255, 160/255, 251/255),  # G1 H
        'silver',                      # G3 S (New)
        'grey',   
        'gold',                      # G2 S
        'magenta'
        ]
)

In [ ]:
gene_num = 40  # how many labels to show on volcano plots
top_n_for_deltas = 30  # how many top |score| genes to use when computing deltas


# ---------- helper to get symmetric-difference "deltas" between up and down ----------

def get_deltas(up_df, down_df, n=20):
    """Return symmetric difference of top-n genes by |score| between up and down."""
    k_up = n // 2
    k_dn = n - k_up

    # Top up-regulated (largest positive scores)
    up_up   = up_df.sort_values('score', ascending=False).head(k_up)['gene']
    down_up = down_df.sort_values('score', ascending=False).head(k_up)['gene']

    # Top down-regulated (most negative scores)
    up_dn   = up_df.sort_values('score', ascending=True).head(k_dn)['gene']
    down_dn = down_df.sort_values('score', ascending=True).head(k_dn)['gene']

    deltas_up   = list(set(up_up) ^ set(down_up))
    deltas_down = list(set(up_dn) ^ set(down_dn))
    return deltas_up + deltas_down


# =========================
# 1) HIPPOCAMPUS: Up vs Down
# =========================

# Volcano plots
ezy.plot.volcano(
    hippo_up_genes_A,
    y_axis='score',
    default_colors=['grey', 'grey'],
    significant_colors=['red', 'blue'],
    balance_labels=True,
    return_highlights=False,
    top_labels=gene_num,
    figsize=(8, 4),
    sig_markers=["+", "D"],
    title='Hippocampus – Stroke > Healthy'
)

ezy.plot.volcano(
    hippo_down_genes_A,
    y_axis='score',
    default_colors=['grey', 'grey'],
    significant_colors=['gold', 'magenta'],
    balance_labels=True,
    return_highlights=False,
    top_labels=gene_num,
    figsize=(8, 4),
    sig_markers=["+", "D"],
    title='Hippocampus – Stroke < Healthy'
)

# Deltas for bars
hippo_deltas = get_deltas(hippo_up_genes_A, hippo_down_genes_A, n=top_n_for_deltas)

# Bar plots (bold genes = different between Up and Down)
ezy.plot.feats_bar(
    hippo_up_genes_A,
    orientation='horizontal',
    bold_genes=hippo_deltas,
    fig_size=(7, 4),
    num_feats=30,  # keep your color style for up
    title='Hippocampus – Up (Stroke > Healthy)'
)

ezy.plot.feats_bar(
    hippo_down_genes_A,
    orientation='horizontal',
    bold_genes=hippo_deltas,
    fig_size=(7, 4),
    num_feats=30,
    colors=['gold', 'magenta'],
    title='Hippocampus – Down (Stroke < Healthy)'
)


# ======================
# 2) CORTEX: Up vs Down
# ======================

# Volcano plots
ezy.plot.volcano(
    cortex_up_genes_A,
    y_axis='score',
    default_colors=['grey', 'grey'],
    significant_colors=['red', 'blue'],
    balance_labels=True,
    return_highlights=False,
    top_labels=gene_num,
    figsize=(8, 4),
    sig_markers=["+", "D"],
    title='Cortex – Stroke > Healthy'
)

ezy.plot.volcano(
    cortex_down_genes_A,
    y_axis='score',
    default_colors=['grey', 'grey'],
    significant_colors=['gold', 'magenta'],
    balance_labels=True,
    return_highlights=False,
    top_labels=gene_num,
    figsize=(8, 4),
    sig_markers=["+", "D"],
    title='Cortex – Stroke < Healthy'
)

# Deltas for bars
cortex_deltas = get_deltas(cortex_up_genes_A, cortex_down_genes_A, n=top_n_for_deltas)

# Bar plots
ezy.plot.feats_bar(
    cortex_up_genes_A,
    orientation='horizontal',
    bold_genes=cortex_deltas,
    fig_size=(7, 4),
    num_feats=30,
    title='Cortex – Up (Stroke > Healthy)'
)

ezy.plot.feats_bar(
    cortex_down_genes_A,
    orientation='horizontal',
    bold_genes=cortex_deltas,
    colors=['gold', 'magenta'],
    fig_size=(7, 4),
    num_feats=30,
    title='Cortex – Down (Stroke < Healthy)'
)


# =========================
# 3) CEREBELLUM: Up vs Down
# =========================

# Volcano plots
ezy.plot.volcano(
    Cerebellum_up_genes_A,
    y_axis='score',
    default_colors=['grey', 'grey'],
    significant_colors=['red', 'blue'],
    balance_labels=True,
    return_highlights=False,
    top_labels=gene_num,
    figsize=(8, 4),
    sig_markers=["+", "D"],
    title='Cerebellum – Stroke > Healthy'
)

ezy.plot.volcano(
    Cerebellum_down_genes_A,
    y_axis='score',
    default_colors=['grey', 'grey'],
    significant_colors=['gold', 'magenta'],
    balance_labels=True,
    return_highlights=False,
    top_labels=gene_num,
    figsize=(8, 4),
    sig_markers=["+", "D"],
    title='Cerebellum – Stroke < Healthy'
)

# Deltas for bars
cereb_deltas = get_deltas(Cerebellum_up_genes_A, Cerebellum_down_genes_A, n=top_n_for_deltas)

# Bar plots
ezy.plot.feats_bar(
    Cerebellum_up_genes_A,
    orientation='horizontal',
    bold_genes=cereb_deltas,
    fig_size=(7, 4),
    num_feats=30,
    title='Cerebellum – Up (Stroke > Healthy)'
)

ezy.plot.feats_bar(
    Cerebellum_down_genes_A,
    orientation='horizontal',
    bold_genes=cereb_deltas,
    fig_size=(7, 4),
    colors=['gold', 'magenta'],
    num_feats=30,
    title='Cerebellum – Down (Stroke < Healthy)'
)
